In [3]:
# Ô 7.0 — Tải hai bộ OTRF và nhìn dữ liệu thô
import sys
from pathlib import Path
import pandas as pd
sys.path.append(str(Path("..").resolve()))
from src import evidence as ev
 
raw = {name: ev.fetch(name) for name in ev.DATASETS}
for name, events in raw.items():
    print(f"{name}: {len(events)} sự kiện")
r = raw["empire_psremoting_stager"]

print("Channel  :", pd.Series([e.get("Channel") for e in r]).value_counts().head(6).to_dict())
print("Hostname :", pd.Series([e.get("Hostname") for e in r]).value_counts().to_dict())

empire_psremoting_stager: 2744 sự kiện
empire_shell_net_localgroup_administrators: 884 sự kiện
Channel  : {'Microsoft-Windows-Sysmon/Operational': 1651, 'security': 544, 'Windows PowerShell': 217, 'Microsoft-Windows-PowerShell/Operational': 173, 'Security': 152, 'System': 6}
Hostname : {'WORKSTATION6.theshire.local': 2003, 'WORKSTATION5.theshire.local': 504, 'WORKSTATION6': 149, 'MORDORDC.theshire.local': 88}


In [5]:
# Ô 7.1 — Chuẩn hóa về một bảng thống nhất
df = pd.concat([ev.normalize(events, name) for name, events in raw.items()], ignore_index=True)
df = ev.attach_pid(df)
print(sum(len(v) for v in raw.values()), "sự kiện thô ->", len(df), "sau chuẩn hóa |",
      df.groupby("channel").size().to_dict())
print("máy:", sorted(df["host"].unique()))
p1 = df[(df["channel"] == "sysmon") & (df["event_id"] == 1)]
print(p1[["host", "pid", "ppid", "image"]].assign(image=p1["image"].str.split("\\").str[-1])
      .to_string(index=False))

3628 sự kiện thô -> 3290 sau chuẩn hóa | {'powershell': 232, 'security': 925, 'sysmon': 2133}
máy: ['mordordc', 'workstation5', 'workstation6']
        host   pid  ppid           image
workstation6  9036   960 wsmprovhost.exe
workstation6 10116  9036  powershell.exe
workstation6  3384 10116     conhost.exe
workstation6  2820   960    WmiPrvSE.exe
workstation6  3148 10116      whoami.exe
workstation5  5956  7456         net.exe
workstation5  8040  5956        net1.exe


In [6]:
# Ô 7.2 — Tạo bảng events chia theo ngày và nạp dữ liệu
conn = ev.connect()
conn.execute("DROP TABLE IF EXISTS events CASCADE")
ev.create_events_table(conn)
ev.insert_events(conn,df)
rows = conn.execute("""
    SELECT tableoid::regclass AS bang_con, count(*), min(ts), max(ts)
    FROM events GROUP BY 1 ORDER BY 1""").fetchall()
for part, n, t_min, t_max in rows:
    print(f"{part} | {n} dòng | {t_min:%Y-%m-%d %H:%M:%S} -> {t_max:%H:%M:%S} UTC")

events_20200920 | 2520 dòng | 2020-09-20 21:08:28 -> 21:10:07 UTC
events_20200921 | 770 dòng | 2020-09-21 23:18:44 -> 23:19:19 UTC


In [7]:
# Ô 7.3 — Thấy Postgres chỉ đọc một bảng con
plan = conn.execute("""
    EXPLAIN SELECT count(*) FROM events
    WHERE ts >= '2020-09-20 00:00+00' AND ts < '2020-09-21 00:00+00'
      AND channel = 'powershell'""").fetchall()
print("\n".join(row[0] for row in plan))

Aggregate  (cost=117.94..117.95 rows=1 width=8)
  ->  Index Only Scan using events_20200920_host_channel_event_id_ts_idx on events_20200920 events  (cost=0.28..117.51 rows=173 width=0)
        Index Cond: ((channel = 'powershell'::text) AND (ts >= '2020-09-20 00:00:00+00'::timestamp with time zone) AND (ts < '2020-09-21 00:00:00+00'::timestamp with time zone))


In [12]:
# Ô 7.4 — Chọn cảnh báo
alerts = conn.execute("""
    SELECT id, ts, host, cmdline FROM events
    WHERE channel = 'sysmon' AND event_id = 1
      AND image ILIKE '%%powershell.exe' AND cmdline ILIKE '%% -enc %%'
    ORDER BY ts""").fetchall()
for a in alerts:
    print(a[0], a[1], a[2], a[3][:80])
ALERT = alerts[0][0]

760 2020-09-20 21:09:12.956000+00:00 workstation6 "C:\windows\System32\WindowsPowerShell\v1.0\powershell.exe" -noP -sta -w 1 -enc 


In [13]:
# Ô 7.5 — Cây tiến trình bằng WITH RECURSIVE
tree = conn.execute(ev.TREE_SQL, dict(alert_id=ALERT, max_up=3, max_down=2,
                                      before_min=10, after_min=5)).fetchall()
for depth, _id, ts, host, pid, image, parent, cmd, user, logon in tree:
    print(f"độ sâu {depth:+d} | {ts:%H:%M:%S} | {ev._name(parent):16s} -> {ev._name(image):16s} "
          f"| PID {pid} | LogonId {logon}")

độ sâu -1 | 21:09:09 | svchost.exe      -> wsmprovhost.exe  | PID 9036 | LogonId 0x2da8840
độ sâu +0 | 21:09:12 | wsmprovhost.exe  -> powershell.exe   | PID 10116 | LogonId 0x2da8840
độ sâu +1 | 21:09:13 | powershell.exe   -> conhost.exe      | PID 3384 | LogonId 0x2da8840
độ sâu +1 | 21:09:40 | powershell.exe   -> whoami.exe       | PID 3148 | LogonId 0x2da8840


In [14]:
# Ô 7.6 — Nối đăng nhập (4624) và script PowerShell (4104)
alert_row = conn.execute("SELECT host, logon_id, ts FROM events WHERE id = %s", (ALERT,)).fetchone()
host, logon_id, ts = alert_row
print(conn.execute(ev.LOGON_SQL, dict(host=host, logon_id=logon_id, ts=ts)).fetchall())
session = conn.execute("""
    SELECT channel, event_id, count(*), sum(length(raw::text)) FROM events
    WHERE host = %s AND (logon_id = %s OR target_logon_id = %s)
    GROUP BY 1, 2 ORDER BY 3 DESC""", (host, logon_id, logon_id)).fetchall()
print("cả phiên", logon_id, "có", sum(r[2] for r in session), "sự kiện,",
      sum(r[3] for r in session), "ký tự JSON thô | nhiều nhất:",
      [(f"{ch} {eid}", n) for ch, eid, n, _ in session[:5]])
pids = [row[4] for row in tree if row[4] is not None]
ps = conn.execute(ev.PS_SQL, dict(host=host, pids=pids, ts=ts, before_min=10, after_min=5,
                                  max_ps=2)).fetchall()
print("PID trong cây:", pids, "| số 4104 khớp:", len(ps), "| độ dài script:", [len(r[4]) for r in ps])
import base64
cmd = conn.execute("SELECT cmdline FROM events WHERE id = %s", (ALERT,)).fetchone()[0]
decoded = base64.b64decode(cmd.split("-enc ")[1].strip()).decode("utf-16-le")
print("giải mã -enc:", len(decoded), "ký tự | script 4104:", len(ps[0][4]), "ký tự")
print(decoded[:150])
print("4103 trên máy này (không lấy):", conn.execute("""
    SELECT count(*) FROM events WHERE host = %s AND channel = 'powershell'
      AND event_id = 4103""", (host,)).fetchone()[0])

[(415, datetime.datetime(2020, 9, 20, 21, 9, 9, 802000, tzinfo=datetime.timezone.utc), 'workstation6', 3, 'THESHIRE.LOCAL\\pgustavo', '0x2da8840', '-')]
cả phiên 0x2da8840 có 450 sự kiện, 830828 ký tự JSON thô | nhiều nhất: [('security 4658', 166), ('security 4656', 94), ('security 4690', 84), ('security 4663', 49), ('security 4674', 44)]
PID trong cây: [9036, 10116, 3384, 3148] | số 4104 khớp: 1 | độ dài script: [1910]
giải mã -enc: 1906 ký tự | script 4104: 1910 ký tự
If($PSVErSioNTaBlE.PSVERsIon.MAJoR -ge 3){$43DE2=[REF].AssEmbly.GeTTYPE('System.Management.Automation.Utils')."GetFIe`Ld"('cachedGroupPolicySettings',
4103 trên máy này (không lấy): 97


In [15]:
# Ô 7.7 — Dựng gói mức 0 và mức 1
for level in [0, 1]:
    pack = ev.build_pack(conn, ALERT, level=level)
    text = ev.render(pack)
    print(f"===== MỨC {level}: {len(pack)} dòng | {len(text)} ký tự ≈ {len(text) // 4} token")
    print(text)
print(pack[["ma", "nguon", "event_row"]].to_string(index=False))

===== MỨC 0: 1 dòng | 324 ký tự ≈ 81 token
[E1] 21:09:12 workstation6: wsmprovhost.exe tạo tiến trình powershell.exe (PID 10116), tài khoản THESHIRE\pgustavo, LogonId 0x2da8840; lệnh: "C:\windows\System32\WindowsPowerShell\v1.0\powershell.exe" -noP -sta -w 1 -enc SQBmACgAJABQAFMAVgBFAHIAUwBpAG8ATgBUAGEAQgBsAEUALgBQAFMAVgBFAFIAcwBJAG8AbgAuAE0A… [rút gọn, 5164 ký tự]
===== MỨC 1: 6 dòng | 1254 ký tự ≈ 313 token
[E1] 21:09:09 workstation6: đăng nhập thành công kiểu 3 (qua mạng) của THESHIRE.LOCAL\pgustavo, LogonId 0x2da8840, IP nguồn -
[E2] 21:09:09 workstation6: svchost.exe tạo tiến trình wsmprovhost.exe (PID 9036), tài khoản THESHIRE\pgustavo, LogonId 0x2da8840; lệnh: C:\windows\system32\wsmprovhost.exe -Embedding
[E3] 21:09:12 workstation6: wsmprovhost.exe tạo tiến trình powershell.exe (PID 10116), tài khoản THESHIRE\pgustavo, LogonId 0x2da8840; lệnh: "C:\windows\System32\WindowsPowerShell\v1.0\powershell.exe" -noP -sta -w 1 -enc SQBmACgAJABQAFMAVgBFAHIAUwBpAG8ATgBUAGEAQgBsAEUALgBQAFMA

In [16]:
# Ô 7.8 — Gói cho cảnh báo thứ hai: khi thiếu nguồn
other = conn.execute("""
    SELECT id FROM events
    WHERE channel = 'sysmon' AND event_id = 1 AND right(lower(image), 8) = '\\net.exe'
    ORDER BY ts LIMIT 1""").fetchone()[0]
print(ev.render(ev.build_pack(conn, other, level=1)))
print("workstation5: số 4624 =", conn.execute(
    "SELECT count(*) FROM events WHERE host = 'workstation5' AND event_id = 4624").fetchone()[0],
    "| PowerShell theo PID:", conn.execute("""
    SELECT exec_pid, event_id, count(*) FROM events
    WHERE host = 'workstation5' AND channel = 'powershell' AND ts >= '2020-09-21 00:00+00'
    GROUP BY 1, 2""").fetchall())

[E1] 23:18:52 workstation5: powershell.exe tạo tiến trình net.exe (PID 5956), tài khoản THESHIRE\pgustavo, LogonId 0x10a7df; lệnh: "C:\windows\system32\net.exe" localgroup Administrators
[E2] 23:18:52 workstation5: net.exe tạo tiến trình net1.exe (PID 8040), tài khoản THESHIRE\pgustavo, LogonId 0x10a7df; lệnh: C:\windows\system32\net1 localgroup Administrators
workstation5: số 4624 = 0 | PowerShell theo PID: [(7456, 4103, 59)]


In [17]:
# Ô 7.9 — Thử các giới hạn
variants = {"mặc định": {}, "max_up=0": dict(max_up=0), "max_down=0": dict(max_down=0),
            "after_min=0": dict(after_min=0), "max_ps=0": dict(max_ps=0),
            "max_lines=4": dict(max_lines=4)}
for name, kw in variants.items():
    pack = ev.build_pack(conn, ALERT, level=1, **kw)
    print(f"{name:12s} | {len(pack)} dòng | {', '.join(pack['nguon'])}")

mặc định     | 6 dòng | đăng nhập, tổ tiên, cảnh báo, PowerShell 4104, con cháu, con cháu
max_up=0     | 5 dòng | đăng nhập, cảnh báo, PowerShell 4104, con cháu, con cháu
max_down=0   | 4 dòng | đăng nhập, tổ tiên, cảnh báo, PowerShell 4104
after_min=0  | 3 dòng | đăng nhập, tổ tiên, cảnh báo
max_ps=0     | 5 dòng | đăng nhập, tổ tiên, cảnh báo, con cháu, con cháu
max_lines=4  | 4 dòng | đăng nhập, tổ tiên, cảnh báo, con cháu


In [18]:
# Ô 7.10 — Múi giờ, tạo trước bảng con, tách bảng con
print(conn.execute("""SELECT '2020-09-21 23:18:44+00'::timestamptz AT TIME ZONE 'Asia/Ho_Chi_Minh'"""
                   ).fetchone()[0], "giờ Việt Nam")
ev.ensure_partition(conn, "2020-09-22")
print([r[0] for r in conn.execute("""
    SELECT inhrelid::regclass::text FROM pg_inherits
    WHERE inhparent = 'events'::regclass ORDER BY 1""").fetchall()])
conn.execute("ALTER TABLE events DETACH PARTITION events_20200921")
print("sau DETACH:", conn.execute("SELECT count(*) FROM events").fetchone()[0], "dòng trong events |",
      conn.execute("SELECT count(*) FROM events_20200921").fetchone()[0], "dòng trong bảng đã tách")
conn.execute("ALTER TABLE events ATTACH PARTITION events_20200921 "
             "FOR VALUES FROM ('2020-09-21 00:00:00+00') TO ('2020-09-22 00:00:00+00')")
print("sau ATTACH:", conn.execute("SELECT count(*) FROM events").fetchone()[0], "dòng")

2020-09-22 06:18:44 giờ Việt Nam
['events_20200920', 'events_20200921', 'events_20200922']
sau DETACH: 2520 dòng trong events | 770 dòng trong bảng đã tách
sau ATTACH: 3290 dòng
